In [9]:
import torch
import torch.nn as nn
import json
import sys
import os
from pathlib import Path
import math

class FLOPOPCalculator:
    """
    Calculate FLOPs for neural network models and OPs for RDDLGN (excluding embeddings)
    """
    
    def __init__(self, config):
        self.config = config
    
    def calculate_rnn_flops(self, input_size, hidden_size, sequence_length, batch_size, num_layers=1):
        """Calculate FLOPs for vanilla RNN layer"""
        flops_per_timestep = 0
        # Input to hidden: W_ih * x_t
        flops_per_timestep += input_size * hidden_size
        # Hidden to hidden: W_hh * h_{t-1}
        flops_per_timestep += hidden_size * hidden_size
        # Bias addition: +b
        flops_per_timestep += hidden_size
        # Tanh activation
        flops_per_timestep += hidden_size
        
        return flops_per_timestep * sequence_length * batch_size * num_layers
    
    def calculate_gru_flops(self, input_size, hidden_size, sequence_length, batch_size, num_layers=1):
        """Calculate FLOPs for GRU layer"""
        flops_per_timestep = 0
        
        # Reset gate: W_ir * x_t + W_hr * h_{t-1} + b_r + sigmoid
        flops_per_timestep += input_size * hidden_size  # W_ir * x_t
        flops_per_timestep += hidden_size * hidden_size  # W_hr * h_{t-1}
        flops_per_timestep += hidden_size  # bias
        flops_per_timestep += hidden_size  # sigmoid
        
        # Update gate: W_iz * x_t + W_hz * h_{t-1} + b_z + sigmoid
        flops_per_timestep += input_size * hidden_size  # W_iz * x_t
        flops_per_timestep += hidden_size * hidden_size  # W_hz * h_{t-1}
        flops_per_timestep += hidden_size  # bias
        flops_per_timestep += hidden_size  # sigmoid
        
        # New gate: W_in * x_t + W_hn * (r_t * h_{t-1}) + b_n + tanh
        flops_per_timestep += input_size * hidden_size  # W_in * x_t
        flops_per_timestep += hidden_size  # r_t * h_{t-1} (element-wise)
        flops_per_timestep += hidden_size * hidden_size  # W_hn * (r_t * h_{t-1})
        flops_per_timestep += hidden_size  # bias
        flops_per_timestep += hidden_size  # tanh
        
        # Final combination: h_t = (1 - z_t) * n_t + z_t * h_{t-1}
        flops_per_timestep += hidden_size  # (1 - z_t)
        flops_per_timestep += hidden_size  # (1 - z_t) * n_t
        flops_per_timestep += hidden_size  # z_t * h_{t-1}
        flops_per_timestep += hidden_size  # final addition
        
        return flops_per_timestep * sequence_length * batch_size * num_layers
    
    def calculate_attention_flops(self, seq_len, d_model, num_heads, batch_size):
        """Calculate FLOPs for multi-head self-attention"""
        d_k = d_model // num_heads
        
        # Q, K, V projections: 3 * (seq_len * d_model * d_model)
        qkv_flops = 3 * seq_len * d_model * d_model * batch_size
        
        # Attention scores: Q * K^T for each head
        attention_scores_flops = batch_size * num_heads * seq_len * seq_len * d_k
        
        # Softmax: approximately 3 operations per element (exp, sum, div)
        softmax_flops = batch_size * num_heads * seq_len * seq_len * 3
        
        # Attention * V
        attention_v_flops = batch_size * num_heads * seq_len * seq_len * d_k
        
        # Output projection: seq_len * d_model * d_model
        output_proj_flops = seq_len * d_model * d_model * batch_size
        
        return qkv_flops + attention_scores_flops + softmax_flops + attention_v_flops + output_proj_flops
    
    def calculate_feedforward_flops(self, seq_len, d_model, d_ff, batch_size):
        """Calculate FLOPs for feed-forward network"""
        return batch_size * seq_len * (d_model * d_ff + d_ff + d_ff * d_model)
    
    def calculate_layer_norm_flops(self, batch_size, seq_len, d_model):
        """Calculate FLOPs for layer normalization"""
        return batch_size * seq_len * d_model * 5
    
    def calculate_linear_flops(self, input_size, output_size, batch_size, sequence_length=1):
        """Calculate FLOPs for linear layer"""
        flops_per_sample = input_size * output_size + output_size
        return flops_per_sample * batch_size * sequence_length
    
    def calculate_positional_encoding_flops(self, batch_size, seq_len, d_model):
        """Calculate FLOPs for positional encoding (just addition)"""
        return batch_size * seq_len * d_model
    
    def calculate_softmax_flops(self, batch_size, sequence_length, vocab_size):
        """Calculate FLOPs for softmax operation"""
        flops_per_position = 3 * vocab_size - 1
        return flops_per_position * batch_size * sequence_length
    
    def calculate_cross_entropy_flops(self, batch_size, sequence_length, vocab_size):
        """Calculate FLOPs for cross-entropy loss"""
        flops_per_position = 2 * vocab_size
        return flops_per_position * batch_size * sequence_length
    
    def calculate_rddlgn_ops(self):
        """Calculate total OPs for RDDLGN model from layer sizes"""
        model_params = self.config['model']['params']
        
        total_ops = 0
        
        # Sum all layer sizes
        for layer_type in ['k_layers_sizes', 'l_layers_sizes', 'm_layers_sizes', 
                          'n_layers_sizes', 'p_layers_sizes']:
            if layer_type in model_params:
                layer_sizes = model_params[layer_type]
                total_ops += sum(layer_sizes)
        
        return total_ops
    
    def calculate_unsynced_rnn_flops(self, src_seq_len, tgt_seq_len, batch_size, mode='training'):
        """Calculate total FLOPs for UnsyncedRNN model (excluding embeddings)"""
        model_params = self.config['model']['params']
        tokenizer_params = self.config['tokenizer']['params']
        
        embedding_dim = model_params['embedding_dim']
        hidden_dim = model_params['hidden_dim']
        num_layers = model_params.get('num_layers', 1)
        vocab_size = tokenizer_params['max_vocab_size']
        
        total_flops = 0
        
        # NOTE: Embedding lookups are excluded from FLOP calculation
        
        # Encoder and decoder RNN (using embedding_dim as input size)
        total_flops += self.calculate_rnn_flops(embedding_dim, hidden_dim, src_seq_len, batch_size, num_layers)
        total_flops += self.calculate_rnn_flops(embedding_dim, hidden_dim, tgt_seq_len, batch_size, num_layers)
        
        # Output linear layer
        total_flops += self.calculate_linear_flops(hidden_dim, vocab_size, batch_size, tgt_seq_len)
        
        if mode == 'training':
            total_flops += self.calculate_softmax_flops(batch_size, tgt_seq_len, vocab_size)
            total_flops += self.calculate_cross_entropy_flops(batch_size, tgt_seq_len, vocab_size)
        
        return total_flops
    
    def calculate_unsynced_gru_flops(self, src_seq_len, tgt_seq_len, batch_size, mode='training'):
        """Calculate total FLOPs for UnsyncedGRU model (excluding embeddings)"""
        model_params = self.config['model']['params']
        tokenizer_params = self.config['tokenizer']['params']
        
        embedding_dim = model_params['embedding_dim']
        hidden_dim = model_params['hidden_dim']
        num_layers = model_params.get('num_layers', 1)
        vocab_size = tokenizer_params['max_vocab_size']
        
        total_flops = 0
        
        # NOTE: Embedding lookups are excluded from FLOP calculation
        
        # Encoder and decoder GRU (using embedding_dim as input size)
        total_flops += self.calculate_gru_flops(embedding_dim, hidden_dim, src_seq_len, batch_size, num_layers)
        total_flops += self.calculate_gru_flops(embedding_dim, hidden_dim, tgt_seq_len, batch_size, num_layers)
        
        # Output linear layer
        total_flops += self.calculate_linear_flops(hidden_dim, vocab_size, batch_size, tgt_seq_len)
        
        if mode == 'training':
            total_flops += self.calculate_softmax_flops(batch_size, tgt_seq_len, vocab_size)
            total_flops += self.calculate_cross_entropy_flops(batch_size, tgt_seq_len, vocab_size)
        
        return total_flops
    
    def calculate_transformer_flops(self, src_seq_len, tgt_seq_len, batch_size, mode='training'):
        """Calculate total FLOPs for Transformer model (excluding embeddings)"""
        model_params = self.config['model']['params']
        tokenizer_params = self.config['tokenizer']['params']
        
        embedding_dim = model_params['embedding_dim']
        hidden_dim = model_params['hidden_dim']  # d_ff
        num_heads = model_params['num_heads']
        num_layers = model_params['num_layers']
        vocab_size = tokenizer_params['max_vocab_size']
        
        total_flops = 0
        
        # NOTE: Embedding lookups are excluded from FLOP calculation
        # Only positional encoding (addition operations)
        total_flops += self.calculate_positional_encoding_flops(batch_size, src_seq_len, embedding_dim)
        total_flops += self.calculate_positional_encoding_flops(batch_size, tgt_seq_len, embedding_dim)
        
        # Encoder layers
        encoder_attention_flops = self.calculate_attention_flops(src_seq_len, embedding_dim, num_heads, batch_size) * num_layers
        encoder_feedforward_flops = self.calculate_feedforward_flops(src_seq_len, embedding_dim, hidden_dim, batch_size) * num_layers
        encoder_layernorm_flops = self.calculate_layer_norm_flops(batch_size, src_seq_len, embedding_dim) * num_layers * 2
        
        total_flops += encoder_attention_flops + encoder_feedforward_flops + encoder_layernorm_flops
        
        # Decoder layers
        decoder_self_attention_flops = self.calculate_attention_flops(tgt_seq_len, embedding_dim, num_heads, batch_size) * num_layers
        decoder_cross_attention_flops = self.calculate_attention_flops(tgt_seq_len, embedding_dim, num_heads, batch_size) * num_layers
        decoder_feedforward_flops = self.calculate_feedforward_flops(tgt_seq_len, embedding_dim, hidden_dim, batch_size) * num_layers
        decoder_layernorm_flops = self.calculate_layer_norm_flops(batch_size, tgt_seq_len, embedding_dim) * num_layers * 3
        
        total_flops += decoder_self_attention_flops + decoder_cross_attention_flops + decoder_feedforward_flops + decoder_layernorm_flops
        
        # Output linear layer
        total_flops += self.calculate_linear_flops(embedding_dim, vocab_size, batch_size, tgt_seq_len)
        
        if mode == 'training':
            total_flops += self.calculate_softmax_flops(batch_size, tgt_seq_len, vocab_size)
            total_flops += self.calculate_cross_entropy_flops(batch_size, tgt_seq_len, vocab_size)
        
        return total_flops
    
    def format_number_latex(self, number):
        """Format numbers in LaTeX-friendly format"""
        if number == 0:
            return "-"
        elif number >= 1e12:
            return f"{number/1e12:.2f}T"
        elif number >= 1e9:
            return f"{number/1e9:.2f}G"
        elif number >= 1e6:
            return f"{number/1e6:.2f}M"
        elif number >= 1e3:
            return f"{number/1e3:.2f}K"
        else:
            return f"{number:.0f}"


def load_config(config_path):
    """Load configuration from JSON file"""
    with open(config_path, 'r') as f:
        return json.load(f)


def find_config_file(filename):
    """Try to find the config file in common locations"""
    current_dir = os.getcwd()
    possible_paths = [
        f"configs/wmt/{filename}",
        f"../configs/wmt/{filename}",
        f"../../configs/wmt/{filename}",
        os.path.join(current_dir, f"configs/wmt/{filename}"),
        filename,  # fallback to current directory
    ]
    
    for path in possible_paths:
        if os.path.exists(path):
            return path
    
    return None


def create_model_configs():
    """Create sample configurations for different models"""
    base_config = {
        "tokenizer": {
            "params": {
                "tokens_per_batch": 1024,
                "seq_length": 16,
                "max_vocab_size": 16000
            }
        },
        "training": {
            "epochs": 1
        }
    }
    
    configs = {
        "RNN": {
            **base_config,
            "model": {
                "name": "UnsyncedRNN",
                "params": {
                    "embedding_dim": 128,
                    "hidden_dim": 128,
                    "num_layers": 1,
                    "dropout": 0.0
                }
            }
        },
        "GRU": {
            **base_config,
            "model": {
                "name": "UnsyncedGRU",
                "params": {
                    "embedding_dim": 128,
                    "hidden_dim": 128,
                    "num_layers": 1,
                    "dropout": 0.0
                }
            }
        },
        "Transformer": {
            **base_config,
            "model": {
                "name": "Transformer",
                "params": {
                    "embedding_dim": 128,
                    "hidden_dim": 512,  # d_ff is typically 4x embedding_dim
                    "num_heads": 8,
                    "num_layers": 6,
                    "dropout": 0.1
                }
            }
        }
    }
    
    return configs


def generate_latex_table(results, src_seq_len, tgt_seq_len, batch_size, vocab_size):
    """Generate LaTeX table code"""
    
    latex_code = """\\begin{table}[!htb]
\\centering
\\setlength{\\tabcolsep}{3mm}
\\begin{sc}
\\begin{tabular}{c|cc}
\\toprule
Model & FLOPs & OPs \\\\
\\midrule"""
    
    # Add data rows
    for model_name in ['Transformer', 'GRU', 'RNN', 'RDDLGN']:
        if model_name in results:
            flops_str = results[model_name]['flops_formatted']
            ops_str = results[model_name]['ops_formatted']
            latex_code += f"\n{model_name} & {flops_str} & {ops_str} \\\\"
    
    latex_code += """
\\bottomrule
\\end{tabular}
\\end{sc}
\\caption{"""
    
    # Generate caption (updated for batch size 1)
    caption = f"""Computational complexity comparison across different model architectures (excluding embedding operations). 
FLOPs (Floating Point Operations) are calculated for a single forward pass with batch size {batch_size}, 
source sequence length {src_seq_len}, target sequence length {tgt_seq_len}, and vocabulary size {vocab_size:,}. 
Embedding lookup operations are excluded from all FLOP calculations to focus on core architectural differences. 
For RNN and GRU models, FLOPs include recurrent computations (matrix multiplications, element-wise operations, 
and activations) and output projections. For Transformer models, FLOPs include positional encodings, 
multi-head attention mechanisms (Q/K/V projections, attention scores, softmax operations), feed-forward networks, 
layer normalizations, and output projections across all encoder and decoder layers. OPs (Operations) for RDDLGN 
represent the sum of all layer sizes from the model configuration, indicating the total number of logical 
operations in the differentiable logic network."""
    
    latex_code += caption
    
    latex_code += """}
\\label{tab:computational_complexity}
\\end{table}"""
    
    return latex_code


def print_console_table(results):
    """Print console version for verification"""
    print("\nCONSOLE TABLE (for verification):")
    print("="*50)
    print(f"{'Model':<15} {'FLOPs':<20} {'OPs':<15}")
    print("-"*50)
    
    for model_name in ['Transformer', 'GRU', 'RNN', 'RDDLGN']:
        if model_name in results:
            flops_str = results[model_name]['flops_formatted']
            ops_str = results[model_name]['ops_formatted']
            print(f"{model_name:<15} {flops_str:<20} {ops_str:<15}")
    
    print("-"*50)


def main():
    print("FLOP/OP LaTeX Table Generator (Batch Size 1, Excluding Embeddings)")
    print("="*70)
    
    # Parameters for FLOP calculations - CHANGED TO BATCH SIZE 1
    src_seq_len = 16
    tgt_seq_len = 16
    batch_size = 1  # Changed from 32 to 1
    vocab_size = 16000
    
    print(f"Analysis parameters:")
    print(f"- Source sequence length: {src_seq_len}")
    print(f"- Target sequence length: {tgt_seq_len}")
    print(f"- Batch size: {batch_size}")
    print(f"- Vocabulary size: {vocab_size:,}")
    print(f"- NOTE: Embedding operations are EXCLUDED from FLOP calculations")
    print()
    
    results = {}
    
    # Create configurations for RNN, GRU, Transformer
    configs = create_model_configs()
    
    # Calculate FLOPs for each standard model
    for model_name, config in configs.items():
        print(f"Calculating FLOPs for {model_name} (excluding embeddings, batch size {batch_size})...")
        calculator = FLOPOPCalculator(config)
        
        if model_name == "RNN":
            flops = calculator.calculate_unsynced_rnn_flops(src_seq_len, tgt_seq_len, batch_size)
        elif model_name == "GRU":
            flops = calculator.calculate_unsynced_gru_flops(src_seq_len, tgt_seq_len, batch_size)
        elif model_name == "Transformer":
            flops = calculator.calculate_transformer_flops(src_seq_len, tgt_seq_len, batch_size)
        
        results[model_name] = {
            'flops': flops,
            'flops_formatted': calculator.format_number_latex(flops),
            'ops': 0,
            'ops_formatted': "-"
        }
        
        print(f"  {model_name} FLOPs: {flops:,} ({calculator.format_number_latex(flops)})")
    
    # Load RDDLGN config and calculate OPs
    rddlgn_config_path = find_config_file("unsynced_recurrent_difflogic.json")
    
    if rddlgn_config_path:
        print(f"Loading RDDLGN config from: {rddlgn_config_path}")
        try:
            rddlgn_config = load_config(rddlgn_config_path)
            calculator = FLOPOPCalculator(rddlgn_config)
            ops = calculator.calculate_rddlgn_ops()
            
            results['RDDLGN'] = {
                'flops': 0,
                'flops_formatted': "-",
                'ops': ops,
                'ops_formatted': calculator.format_number_latex(ops)
            }
            
            print(f"RDDLGN OPs calculated: {ops:,}")
            
        except Exception as e:
            print(f"Error loading RDDLGN config: {e}")
            results['RDDLGN'] = {
                'flops': 0,
                'flops_formatted': "-",
                'ops': 0,
                'ops_formatted': "Error"
            }
    else:
        print("RDDLGN config file not found. Using provided config...")
        # Use the provided config from the user
        rddlgn_config = {
            "model": {
                "params": {
                    "k_layers_sizes": [54000, 32000],
                    "l_layers_sizes": [12000, 12000],
                    "m_layers_sizes": [400000, 400000, 480000],
                    "n_layers_sizes": [12000, 12000],
                    "p_layers_sizes": [64000, 48000]
                }
            }
        }
        
        calculator = FLOPOPCalculator(rddlgn_config)
        ops = calculator.calculate_rddlgn_ops()
        
        results['RDDLGN'] = {
            'flops': 0,
            'flops_formatted': "-",
            'ops': ops,
            'ops_formatted': calculator.format_number_latex(ops)
        }
        
        print(f"RDDLGN OPs calculated from provided config: {ops:,} ({calculator.format_number_latex(ops)})")
    
    # Print console version for verification
    print_console_table(results)
    
    # Generate and print LaTeX code
    latex_table = generate_latex_table(results, src_seq_len, tgt_seq_len, batch_size, vocab_size)
    
    print("\n" + "="*80)
    print("LATEX TABLE CODE:")
    print("="*80)
    print(latex_table)
    print("="*80)
    
    # Save to file
    with open("computational_complexity_table_batch1.tex", "w") as f:
        f.write(latex_table)
    
    print(f"\nLaTeX code saved to: computational_complexity_table_batch1.tex")


if __name__ == "__main__":
    main()

FLOP/OP LaTeX Table Generator (Batch Size 1, Excluding Embeddings)
Analysis parameters:
- Source sequence length: 16
- Target sequence length: 16
- Batch size: 1
- Vocabulary size: 16,000
- NOTE: Embedding operations are EXCLUDED from FLOP calculations

Calculating FLOPs for RNN (excluding embeddings, batch size 1)...
  RNN FLOPs: 35,360,752 (35.36M)
Calculating FLOPs for GRU (excluding embeddings, batch size 1)...
  GRU FLOPs: 37,494,768 (37.49M)
Calculating FLOPs for Transformer (excluding embeddings, batch size 1)...
  Transformer FLOPs: 80,044,016 (80.04M)
Loading RDDLGN config from: ../configs/wmt/unsynced_recurrent_difflogic.json
RDDLGN OPs calculated: 1,526,000

CONSOLE TABLE (for verification):
Model           FLOPs                OPs            
--------------------------------------------------
Transformer     80.04M               -              
GRU             37.49M               -              
RNN             35.36M               -              
RDDLGN          -        